In [1]:
import sys
import os

# Add the parent directory (src) to the system path
# The '..' tells it to look one folder up from where the notebook is currently running
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
#Loadin the rag
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [3]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [4]:
#Judge Instructions
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [5]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [6]:
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress
from client import client2



In [7]:
#check a record
rec = answers[0]
rec

{'question': 'Is it too late to enroll in the course if I just found out about it?',
 'answer_llm': 'Yes, you can still join the course, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [11]:
#Evaluation of the answers using the LLM as a judge
def evaluate_aqa(question, answer_orig, answer_llm, model="openai/gpt-oss-20b"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        client2,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [12]:
#Test it on one record
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer correctly states that enrollment is still possible and that a certificate requires the project submission before the acceptance period ends, matching the key points of the original answer.', score='good')

In [13]:
#run it for all
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [14]:
#parallelize the evaluation for all records
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/515 [00:00<?, ?it/s]

In [15]:
#Split the results and usage
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [16]:
#create a dataframe for the evaluations
df_eval = pd.DataFrame(evaluations)

In [17]:
calc_total_price(usages)

0.14694735

In [18]:
#Check the result
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 469/515 = 91.07%


In [19]:
#Look at the bad cases
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
2,What do I need to do to receive a certificate ...,74eb249bbf,bad,The AI answer adds additional requirements (li...
3,Are there any deadlines I should be aware of f...,74eb249bbf,bad,The AI answer does not provide any information...
4,How does the project submission affect my abil...,74eb249bbf,bad,The original answer states only that submittin...
13,What should I do if I have questions during th...,489dd1c9d9,bad,The AI answer covers the recommendation to use...
24,Do I have to use my real name for the course c...,c2903069a0,bad,The AI answer omits the requirement that the s...


In [20]:
#save the results
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)